In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/industrial-machine-condition-classification-mtech/sample_submission.csv
/kaggle/input/competitions/industrial-machine-condition-classification-mtech/student_dataset.csv
/kaggle/input/competitions/industrial-machine-condition-classification-mtech/test.csv


In [2]:

# ============================================================
# MACHINE CONDITION CLASSIFICATION
# CatBoost + RandomizedSearchCV + 5-Fold Cross-Validation
# Classes: 0 = Normal, 1 = Thermal Failure, 2 = Mechanical Failure
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd

from pathlib import Path
from catboost import CatBoostClassifier

from sklearn.base import clone
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")

SEED = 42

# ------------------------------------------------------------
# 1. FIND AND LOAD THE DATASETS
# ------------------------------------------------------------

DATA_DIR = Path(
    "/kaggle/input/competitions/"
    "industrial-machine-condition-classification-mtech"
)

def find_csv(filename):
    # First try the expected competition directory
    direct_path = DATA_DIR / filename

    if direct_path.exists():
        return direct_path

    # Otherwise search the Kaggle input directory
    matches = list(Path("/kaggle/input").rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename} under /kaggle/input. "
            "Check the competition dataset path."
        )

    return matches[0]


train_path = find_csv("student_dataset.csv")
test_path = find_csv("test.csv")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print("Training dataset shape:", train.shape)
print("Test dataset shape:", test.shape)
print("\nTraining columns:", train.columns.tolist())

TARGET = "condition"
ID_COL = "unit_id"

if TARGET not in train.columns:
    raise ValueError(f"Target column '{TARGET}' not found in training data.")

if ID_COL not in train.columns or ID_COL not in test.columns:
    raise ValueError(f"'{ID_COL}' must exist in both train and test data.")

print("\nClass distribution:")
print(train[TARGET].value_counts().sort_index())

# ------------------------------------------------------------
# 2. SEPARATE FEATURES, TARGET, AND TEST IDS
# ------------------------------------------------------------

y = train[TARGET].astype(int)

if not set(y.unique()).issubset({0, 1, 2}):
    raise ValueError("Unexpected target labels. Expected only 0, 1, and 2.")

test_ids = test[ID_COL].copy()

X = train.drop(columns=[TARGET, ID_COL], errors="ignore").copy()
X_test = test.drop(columns=[TARGET, ID_COL], errors="ignore").copy()

# ------------------------------------------------------------
# 3. FEATURE ENGINEERING
# ------------------------------------------------------------

def add_features(df):
    df = df.copy()

    # Temperature difference
    if {"process_temp", "ambient_temp"}.issubset(df.columns):
        df["temp_diff"] = (
            df["process_temp"] - df["ambient_temp"]
        )

    # Approximate rotational power-related interaction
    if {"drive_torque", "shaft_speed"}.issubset(df.columns):
        df["torque_speed_product"] = (
            df["drive_torque"] * df["shaft_speed"]
        )

    # Interaction between component wear and torque
    if {"component_wear", "drive_torque"}.issubset(df.columns):
        df["wear_torque"] = (
            df["component_wear"] * df["drive_torque"]
        )

    # Temperature difference relative to coolant flow
    if {"process_temp", "ambient_temp", "coolant_flow"}.issubset(df.columns):
        denominator = df["coolant_flow"].replace(0, np.nan)

        df["temp_diff_coolant_ratio"] = (
            (df["process_temp"] - df["ambient_temp"])
            / denominator
        )

    return df


X = add_features(X)
X_test = add_features(X_test)

# Make the training and test feature columns identical
X_test = X_test.reindex(columns=X.columns)

# Explicitly treat unit_type as categorical, if present
if "unit_type" in X.columns:
    X["unit_type"] = X["unit_type"].fillna("__MISSING__").astype(str)
    X_test["unit_type"] = (
        X_test["unit_type"].fillna("__MISSING__").astype(str)
    )

# Detect categorical features
cat_cols = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

# CatBoost categorical columns should not contain missing values
for col in cat_cols:
    X[col] = X[col].fillna("__MISSING__").astype(str)
    X_test[col] = X_test[col].fillna("__MISSING__").astype(str)

# Convert all remaining features to numeric
for col in X.columns:
    if col not in cat_cols:
        X[col] = pd.to_numeric(X[col], errors="coerce")
        X_test[col] = pd.to_numeric(X_test[col], errors="coerce")

# Replace infinite feature values with missing values
X = X.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

# IMPORTANT FIX:
# Keep cat_features OUT of the CatBoostClassifier constructor.
# Pass categorical feature indices to .fit() instead.
cat_feature_indices = [
    X.columns.get_loc(col) for col in cat_cols
]

print("\nNumber of features:", X.shape[1])
print("Categorical columns:", cat_cols)
print("Categorical feature indices:", cat_feature_indices)

# ------------------------------------------------------------
# 4. SET UP 5-FOLD STRATIFIED CROSS-VALIDATION
# ------------------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

# ------------------------------------------------------------
# 5. BASE CATBOOST MODEL
# ------------------------------------------------------------

base_model = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="Accuracy",
    random_seed=SEED,
    verbose=0,
    allow_writing_files=False,
    thread_count=-1
)

# ------------------------------------------------------------
# 6. HYPERPARAMETER SEARCH
# ------------------------------------------------------------

param_distributions = {
    "iterations": [800, 1200, 1600, 2000],
    "depth": [4, 5, 6, 7, 8],
    "learning_rate": [0.02, 0.03, 0.05, 0.07],
    "l2_leaf_reg": [3, 5, 7, 9, 12],
    "random_strength": [0.2, 0.5, 1.0, 2.0],
    "bagging_temperature": [0.0, 0.5, 1.0, 2.0],
    "auto_class_weights": ["Balanced", "SqrtBalanced"]
}

search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_distributions,
    n_iter=30,
    scoring="accuracy",
    cv=cv,
    random_state=SEED,
    n_jobs=1,
    verbose=2,
    refit=True,
    return_train_score=True,
    error_score="raise"
)

print("\nStarting hyperparameter search...")
print("Configurations:", search.n_iter)
print("Cross-validation folds:", cv.n_splits)
print("Maximum model fits during search:", search.n_iter * cv.n_splits)

# FIX: Supply categorical feature indices here
search.fit(
    X,
    y,
    cat_features=cat_feature_indices
)

# ------------------------------------------------------------
# 7. DISPLAY THE BEST SEARCH RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("BEST HYPERPARAMETERS")
print("=" * 60)

print(search.best_params_)

print("\nBest cross-validation accuracy: "
      f"{search.best_score_ * 100:.2f}%")

results = pd.DataFrame(search.cv_results_)

top_results = results.sort_values(
    by="mean_test_score",
    ascending=False
)[
    [
        "mean_train_score",
        "mean_test_score",
        "std_test_score",
        "params"
    ]
].head(5)

print("\nTop 5 configurations:")
print(top_results.to_string(index=False))

# ------------------------------------------------------------
# 8. OUT-OF-FOLD EVALUATION
# ------------------------------------------------------------

# Evaluate each row using a model that did not train on that row.
oof_predictions = np.empty(len(X), dtype=int)
fold_accuracies = []

print("\n" + "=" * 60)
print("OUT-OF-FOLD EVALUATION")
print("=" * 60)

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
    X_fold_train = X.iloc[train_idx]
    y_fold_train = y.iloc[train_idx]

    X_fold_valid = X.iloc[valid_idx]
    y_fold_valid = y.iloc[valid_idx]

    fold_model = clone(base_model)
    fold_model.set_params(**search.best_params_)

    # FIX: Pass categorical indices to every model's .fit()
    fold_model.fit(
        X_fold_train,
        y_fold_train,
        cat_features=cat_feature_indices
    )

    fold_predictions = np.asarray(
        fold_model.predict(X_fold_valid)
    ).reshape(-1).astype(int)

    oof_predictions[valid_idx] = fold_predictions

    fold_accuracy = accuracy_score(
        y_fold_valid,
        fold_predictions
    )

    fold_accuracies.append(fold_accuracy)

    print(f"Fold {fold} accuracy: {fold_accuracy * 100:.2f}%")

oof_accuracy = accuracy_score(y, oof_predictions)
mean_fold_accuracy = np.mean(fold_accuracies)
std_fold_accuracy = np.std(fold_accuracies, ddof=1)

print("\nOverall out-of-fold accuracy: "
      f"{oof_accuracy * 100:.2f}%")

print("Mean fold accuracy: "
      f"{mean_fold_accuracy * 100:.2f}%")

print("Standard deviation across folds: "
      f"{std_fold_accuracy * 100:.2f} percentage points")

print("\nClassification report:")
print(
    classification_report(
        y,
        oof_predictions,
        labels=[0, 1, 2],
        target_names=[
            "Normal (0)",
            "Thermal Failure (1)",
            "Mechanical Failure (2)"
        ],
        zero_division=0
    )
)

print("Confusion matrix (actual rows, predicted columns):")
print(confusion_matrix(y, oof_predictions, labels=[0, 1, 2]))

# ------------------------------------------------------------
# 9. TRAIN THE FINAL MODEL AND PREDICT TEST DATA
# ------------------------------------------------------------

# RandomizedSearchCV with refit=True has already fitted
# best_estimator_ on the full training dataset.
best_model = search.best_estimator_

test_predictions = np.asarray(
    best_model.predict(X_test)
).reshape(-1).astype(int)

# ------------------------------------------------------------
# 10. CREATE THE KAGGLE SUBMISSION FILE
# ------------------------------------------------------------

submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: test_predictions
})

output_path = "/kaggle/working/submission_catboost_tuned.csv"
submission.to_csv(output_path, index=False)

# ------------------------------------------------------------
# 11. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FINAL ACCURACY SUMMARY")
print("=" * 60)

print(f"Best search CV accuracy: {search.best_score_ * 100:.2f}%")
print(f"Overall OOF accuracy:    {oof_accuracy * 100:.2f}%")
print(f"Mean fold accuracy:      {mean_fold_accuracy * 100:.2f}%")
print(f"Fold accuracy std dev:   {std_fold_accuracy * 100:.2f}%")

if oof_accuracy >= 0.90:
    print("OOF accuracy reached 90%.")
else:
    print(f"OOF accuracy is {90 - oof_accuracy * 100:.2f} percentage "
          "points below 90%.")

print("\nSubmission rows:", len(submission))
print("Submission columns:", submission.columns.tolist())
print("Predicted labels:", sorted(submission[TARGET].unique().tolist()))
print("Saved submission:", output_path)

print("\nSubmission preview:")
print(submission.head())

print("\nIMPORTANT:")
print("- CV/OOF accuracy is an estimate from training data.")
print("- It is not a guarantee of your hidden-test leaderboard accuracy.")
print("- Hyperparameter selection can make these estimates optimistic.")
print("- Check the submission columns before submitting on Kaggle.")


Training dataset shape: (16000, 13)
Test dataset shape: (4000, 12)

Training columns: ['unit_id', 'unit_type', 'ambient_temp', 'process_temp', 'coolant_flow', 'shaft_speed', 'drive_torque', 'power_load', 'component_wear', 'thermal_stress', 'mechanical_load', 'vibration_index', 'condition']

Class distribution:
condition
0    8000
1    4000
2    4000
Name: count, dtype: int64

Number of features: 15
Categorical columns: ['unit_type']
Categorical feature indices: [0]

Starting hyperparameter search...
Configurations: 30
Cross-validation folds: 5
Maximum model fits during search: 150
Fitting 5 folds for each of 30 candidates, totalling 150 fits
[CV] END auto_class_weights=SqrtBalanced, bagging_temperature=0.0, depth=6, iterations=1600, l2_leaf_reg=12, learning_rate=0.03, random_strength=1.0; total time=  28.8s
[CV] END auto_class_weights=SqrtBalanced, bagging_temperature=0.0, depth=6, iterations=1600, l2_leaf_reg=12, learning_rate=0.03, random_strength=1.0; total time=  28.0s
[CV] END aut